# Tier 3 — controle de 2 mil: SAINT

SAINT treinado em 2 mil amostras sorteadas do treino (mesmo teste de 50 mil do Tier 3), lote 256.
Seleção: Optuna TPE, 20 trials, hold-out 80/20 nas 2 mil, paciência 16. Final: 10 sementes, 200 épocas
**sem paciência** (com paciência o SAINT parava no platô inicial e previa uma classe só).
Os FT-Softmax/Top-k/CUR rodam localmente; só o SAINT não cabe na GPU local (COVER).

**Como rodar:** Settings → Accelerator → **GPU T4 x2**, Internet **ligada** → **Save Version → Save & Run All**.
Tudo fica em `/kaggle/working/tier3_out/` (só KB: o clone vai para `/tmp`). **Não cancele**: o log aparece ao vivo e a sessão termina sozinha. Se cair no meio, nova versão com a saída como entrada: retoma por semente.

**Depois:** me passe a saída da última célula (ou o `tier3_out/`).

In [ ]:
# ── 1. GPUs ──
import torch
print('CUDA:', torch.cuda.is_available(), '| placas:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()): print(' ', i, torch.cuda.get_device_name(i))
assert torch.cuda.device_count() >= 2, 'Settings → Accelerator → GPU T4 x2'

In [ ]:
# ── 2. Repositório ──
import os, subprocess, sys
GIT_URL     = 'https://github.com/PauloBernardo/dissertacao-estudo-comparativo.git'
# clone FORA de /kaggle/working: lá tudo vira saída (parquets + .git); a saída fica só com tier3_out/
PROJECT_DIR = '/tmp/sparse-lssvm-transformers-study'
BRANCH      = 'revisao/estatistica-e-proveniencia'
if os.path.exists(PROJECT_DIR):
    subprocess.run(['git', '-C', PROJECT_DIR, 'fetch', 'origin'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, GIT_URL, PROJECT_DIR], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'checkout', BRANCH], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--rebase', 'origin', BRANCH], check=True)
os.chdir(PROJECT_DIR)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)
sys.path.insert(0, '.')
assert 'TIER3_DEADLINE' in open('scripts/run_tier3.py').read(), 'clone sem prazo no runner — refaça o pull'

In [ ]:
# ── 3. Dependências e dados (em cache no clone; nada é baixado) ──
subprocess.run('pip install -q einops optuna', shell=True)
from pathlib import Path
from src.data.loaders import DatasetLoader
for d in ['HIGGS500K', 'COVER', 'SUSY']:
    assert Path(f'data/raw/{d}.parquet').exists(), f'sem {d}.parquet no clone — refaça o pull'
    X, y, _ = DatasetLoader.load(d)
    print(f'{d:10s} {X.shape}  positivos {y.mean():.3f}')

In [ ]:
# ── 4. Configuração, retomada automática e execução nas duas placas ──
import json, shutil, time
from pathlib import Path

DEADLINE_H = 8.0     # a sessão deve TERMINAR sozinha: só assim o Kaggle salva a saída (não cancele)
os.environ['TIER3_DEADLINE'] = str(time.time() + DEADLINE_H * 3600)

OUTDIR = Path('/kaggle/working/tier3_out'); OUTDIR.mkdir(exist_ok=True)
# Retomada: copia qualquer tier3_* das entradas (saída de uma sessão anterior
# adicionada via "Add Input" → Notebook Output). Nada a configurar.
for f in sorted(Path('/kaggle/input').rglob('tier3_SAINTColnorm_2k_*')):   # só deste notebook
    if f.is_file() and not (OUTDIR / f.name).exists():
        shutil.copy(f, OUTDIR / f.name); print('retomado:', f)

# Controle de 2 mil do SAINT: uma base por placa (o COVER, com 54 features, é o mais pesado).
SPLIT = {0: 'HIGGS500K SUSY', 1: 'COVER'}

import threading
def run_2gpu(make_cmd, name):
    """Um processo por placa; cada linha vai para o log em tier3_out/ E para a tela."""
    def pump(gpu, proc, log):
        for line in proc.stdout:
            log.write(line); log.flush(); print(f'[gpu{gpu}] {line}', end='', flush=True)
    procs = []
    for gpu in (0, 1):
        log = open(OUTDIR / f'tier3_{name}_gpu{gpu}.log', 'a')
        cmd = f'CUDA_VISIBLE_DEVICES={gpu} python -u ' + make_cmd(gpu)
        print(f'[gpu{gpu}]', cmd, flush=True)
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        t = threading.Thread(target=pump, args=(gpu, p, log)); t.start(); procs.append((gpu, p, t, log))
    for g, p, t, log in procs:
        p.wait(); t.join(); log.close(); print(f'gpu{g} terminou com código {p.returncode}')


In [ ]:
# ── 5. SAINT com 2 mil: seleção (20 trials, paciência 16) + final (10 sementes, 200 épocas sem paciência) ──
run_2gpu(lambda g: 'scripts/tier3_dense_2k.py --variant SAINTColnorm --final-no-patience '
         f'--datasets {SPLIT[g]} --output {OUTDIR}/tier3_SAINTColnorm_2k_gpu{g}.json', 'saint2k')

In [ ]:
# ── 6. Resultado ──
import numpy as np
for g in (0, 1):
    f = OUTDIR / f'tier3_SAINTColnorm_2k_gpu{g}.json'
    if not f.exists(): print('sem', f.name); continue
    d = json.load(open(f))
    for ds in SPLIT[g].split():
        v = [r['test_f1_macro'] for r in d['final'] if r['dataset'] == ds]
        print(f'{ds:10s} n={len(v):2d}  F1 = {np.mean(v):.4f} ± {np.std(v, ddof=1) if len(v) > 1 else 0:.4f}'
              f'  mín {min(v) if v else float("nan"):.3f}  {d["selection"].get(ds, {}).get("params")}')